In [21]:
from pynq import MMIO, Overlay
import time

# RHD SPI IP 的基地址 (根據我們的設計)
RHD_SPI_BASE = 0xA0010000
ADDRESS_RANGE = 0x10000

# 暫存器偏移
CONTROL_REG = 0x00  # bit[0]=start_pulse, bit[1]=stop_pulse, bit[7:4]=phase_select
STATUS_REG  = 0x04  # bit[0]=busy, bit[1]=finish
TX_DATA_REG = 0x08  # 16-bit data to transmit
RX_DATA_REG = 0x0C  # 16-bit received data

# 初始化 MMIO
mmio = MMIO(RHD_SPI_BASE, ADDRESS_RANGE)

# 讀取狀態
status = mmio.read(STATUS_REG)
print(f"SPI Status: 0x{status:08x}")
print(f"Busy: {status & 0x1}")
print(f"Finish: {(status >> 1) & 0x1}")

SPI Status: 0x00000001
Busy: 1
Finish: 0


In [22]:
from pynq import DefaultIP

class RHD_SPI_Driver(DefaultIP):
    """RHD SPI IP 驅動程式"""
    
    def __init__(self, description):
        super().__init__(description=description)
        
    # 綁定到我們的 IP 核心 (VLNV 格式)
    bindto = ['ntk.com:user:rhd_spi_v1_0:1.0']
    
    # 暫存器偏移
    CONTROL_REG = 0x00
    STATUS_REG  = 0x04
    TX_DATA_REG = 0x08
    RX_DATA_REG = 0x0C
    
    def start_transfer(self, tx_data=0, phase_select=0):
        """開始 SPI 傳輸"""
        # 寫入傳輸資料
        self.write(self.TX_DATA_REG, tx_data & 0xFFFF)
        
        # 設定控制暫存器：phase_select + start_pulse
        control_val = ((phase_select & 0xF) << 4) | 0x1
        self.write(self.CONTROL_REG, control_val)
        
        # 清除 start_pulse
        control_val = (phase_select & 0xF) << 4
        self.write(self.CONTROL_REG, control_val)
    
    def wait_completion(self, timeout=1000):
        """等待傳輸完成"""
        for _ in range(timeout):
            status = self.read(self.STATUS_REG)
            if (status >> 1) & 0x1:  # finish bit
                return True
            time.sleep(0.001)
        return False
    
    def read_data(self):
        """讀取接收資料"""
        return self.read(self.RX_DATA_REG) & 0xFFFF
    
    def get_status(self):
        """獲取狀態"""
        status = self.read(self.STATUS_REG)
        return {
            'busy': status & 0x1,
            'finish': (status >> 1) & 0x1,
            'raw': status
        }
    
    def spi_transfer(self, tx_data, phase_select=0, timeout=1000):
        """完整的 SPI 傳輸"""
        # 開始傳輸
        self.start_transfer(tx_data, phase_select)
        
        # 等待完成
        if not self.wait_completion(timeout):
            raise TimeoutError("SPI transfer timeout")
        
        # 讀取結果
        rx_data = self.read_data()
        return rx_data

# RHD2000 系列晶片的命令
class RHD2000_Commands:
    """RHD2000 晶片命令定義"""
    
    # 標準命令
    CALIBRATE_ADC = 0x5500
    CLEAR_SAMPLES = 0x5A00
    READ_REG = lambda reg_addr: 0xC000 | (reg_addr << 8)
    WRITE_REG = lambda reg_addr, data: 0x8000 | (reg_addr << 8) | data
    
    # 常用暫存器地址
    ADC_CONFIG = 0x00
    SUPPLY_SENSOR = 0x01
    ADC_OUTPUT_FORMAT = 0x02
    IMPEDANCE_CTRL = 0x03
    BANDWIDTH_SELECT = 0x04

In [25]:
# 載入 overlay
overlay = Overlay('./rhd_spi.bit')

# 檢查可用的 IP
overlay?

# 如果我們的驅動類別已註冊，直接使用
rhd_spi = overlay.rhd_spi_0  # 根據實際的 IP 名稱

# 或者使用 MMIO 直接存取
if not hasattr(overlay, 'rhd_spi_0'):
    from pynq import MMIO
    rhd_spi = MMIO(0xA0010000, 0x10000)

In [26]:
# 基本 SPI 測試
def test_rhd_spi():
    """測試 RHD SPI 功能"""
    
    print("RHD SPI IP 測試開始...")
    
    # 檢查初始狀態
    status = rhd_spi.get_status()
    print(f"初始狀態: {status}")
    
    # 發送校準命令
    print("發送 ADC 校準命令...")
    rx_data = rhd_spi.spi_transfer(RHD2000_Commands.CALIBRATE_ADC)
    print(f"接收資料: 0x{rx_data:04x}")
    
    # 讀取暫存器
    print("讀取 ADC 配置暫存器...")
    rx_data = rhd_spi.spi_transfer(RHD2000_Commands.READ_REG(0x00))
    print(f"ADC 配置: 0x{rx_data:04x}")
    
    print("測試完成!")

# 執行測試
test_rhd_spi()

RHD SPI IP 測試開始...
初始狀態: {'busy': 1, 'finish': 0, 'raw': 1}
發送 ADC 校準命令...
接收資料: 0x0000
讀取 ADC 配置暫存器...
ADC 配置: 0x0000
測試完成!


In [27]:
def initialize_rhd2000():
    """初始化 RHD2000 晶片"""
    
    print("RHD2000 初始化開始...")
    
    # 1. ADC 校準
    rhd_spi.spi_transfer(RHD2000_Commands.CALIBRATE_ADC)
    time.sleep(0.01)
    
    # 2. 設定 ADC 配置
    rhd_spi.spi_transfer(RHD2000_Commands.WRITE_REG(0x00, 0x96))  # ADC 配置
    
    # 3. 設定頻寬
    rhd_spi.spi_transfer(RHD2000_Commands.WRITE_REG(0x04, 0x03))  # 1kHz-10kHz
    
    # 4. 設定輸出格式
    rhd_spi.spi_transfer(RHD2000_Commands.WRITE_REG(0x02, 0x04))  # 偏移二進制
    
    # 5. 驗證設定
    adc_config = rhd_spi.spi_transfer(RHD2000_Commands.READ_REG(0x00))
    bandwidth = rhd_spi.spi_transfer(RHD2000_Commands.READ_REG(0x04))
    output_format = rhd_spi.spi_transfer(RHD2000_Commands.READ_REG(0x02))
    
    print(f"ADC 配置: 0x{adc_config:04x}")
    print(f"頻寬設定: 0x{bandwidth:04x}")
    print(f"輸出格式: 0x{output_format:04x}")
    
    print("RHD2000 初始化完成!")

# 執行初始化
initialize_rhd2000()

RHD2000 初始化開始...
ADC 配置: 0x0000
頻寬設定: 0x0000
輸出格式: 0x0000
RHD2000 初始化完成!


In [28]:
def collect_data(num_samples=1000):
    """採集 RHD2000 資料"""
    
    print(f"開始採集 {num_samples} 個樣本...")
    
    samples = []
    
    for i in range(num_samples):
        # 發送轉換命令 (通道 0)
        rx_data = rhd_spi.spi_transfer(0x0000, phase_select=0)
        
        # 提取 ADC 資料 (假設為 16 位)
        adc_value = rx_data & 0xFFFF
        samples.append(adc_value)
        
        # 進度顯示
        if (i + 1) % 100 == 0:
            print(f"已採集 {i + 1} 個樣本")
        
        time.sleep(0.001)  # 1ms 間隔
    
    print("資料採集完成!")
    return samples

# 採集資料並分析
samples = collect_data(500)

# 基本統計
import numpy as np
mean_val = np.mean(samples)
std_val = np.std(samples)
print(f"平均值: {mean_val:.2f}")
print(f"標準差: {std_val:.2f}")
print(f"範圍: {min(samples)} - {max(samples)}")

開始採集 500 個樣本...
已採集 100 個樣本
已採集 200 個樣本
已採集 300 個樣本
已採集 400 個樣本
已採集 500 個樣本
資料採集完成!
平均值: 0.00
標準差: 0.00
範圍: 0 - 0


In [2]:
from pynq import Overlay, MMIO
import pynq.lib as lib
from pynq import allocate
import numpy as np

# 載入 overlay
overlay = Overlay('./rhd_spi.bit')
# RHD SPI IP 的基地址 
RHD_SPI_BASE = 0xA0000000
ADDRESS_RANGE = 0x10000

# 暫存器偏移
CONTROL_REG = 0x00  # bit[0]=start_pulse, bit[1]=stop_pulse, bit[7:4]=phase_select
PHA_SEL_REG = 0x04  # [3:0]
TX_DATA_REG = 0x08  # 16-bit data to transmit
RX_DATA_REG = 0x0C  # 16-bit received data
STATUS_REG  = 0x10  # bit[0]=busy, bit[1]=finish
# 初始化 MMIO
mmio = MMIO(RHD_SPI_BASE, ADDRESS_RANGE)

# 讀取狀態
status = mmio.read(STATUS_REG)
print(f"SPI Status: 0x{status:08x}")
print(f"Busy: {status & 0x1}")
print(f"Finish: {(status >> 1) & 0x1}")

SPI Status: 0x000000c5
Busy: 1
Finish: 0


In [3]:
# AXI 地址
CTRL    = 0x00
STATUS  = 0x10
RX_DATA = 0x0C
PH_SEL  = 0x04

# 1️⃣ 送 start_pulse：bit0 = 1
mmio.write(PH_SEL, 0x1)
mmio.write(CTRL, 0x1)

# 2️⃣ 等 BUSY 變 0（或 FINISH 變 1）
while mmio.read(STATUS) & 0x1:        # BUSY 還在
    pass

# 3️⃣ 此時 FINISH = 1，讀 RX_DATA
data = mmio.read(RX_DATA) & 0xFFFF
print(f"RX = 0x{data:04X}")

KeyboardInterrupt: 